In [1]:
import os

In [2]:
%pwd

'c:\\1Kumawat\\Online_learning\\Projects_uploaded_git\\datascience_project_1\\reseach'

In [3]:
os.chdir("../")
%pwd

'c:\\1Kumawat\\Online_learning\\Projects_uploaded_git\\datascience_project_1'

In [4]:
from dataclasses import dataclass
from pathlib import Path 

# dataclasses: a way to define classes that are primarily used to store data (don't have much logic and self-contained)
# classes are used to define the structure of an object
@dataclass
class DataIngestionConfig:
    # define the attributes of the class
    # Path is a class from pathlib module that represents a filesystem path (tak same as config.yaml file)
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [5]:
%pwd

'c:\\1Kumawat\\Online_learning\\Projects_uploaded_git\\datascience_project_1'

In [6]:
from src.data_science_project_1.constants import * # constants.py is a module that contains constants used in the project
from src.data_science_project_1.utils.common import read_yaml, create_directories

In [14]:
class ConfigurationManager:
    def __init__(self,
                 config_file_path= CONFIG_FILE_PATH, # CONFIG_FILE_PATH is a constant that contains the path to the config.yaml file
                 params_file_path=PARAMS_FILE_PATH,
                 schema_file_path=SCHEMA_FILE_PATH
                 ):
        self.config = read_yaml(config_file_path) # read_yaml is a function that reads a YAML file and returns the data as a dictionary
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_roots]) # artifacts_root is the root directory where all the artifacts will be stored

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion # data_ingestion is a key in the config dictionary that contains the data ingestion configuration
        create_directories([config.root_dir]) # create the root directory for data ingestion if it

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir, # root_dir is the directory where the data will be stored
            source_URL=config.source_URL,   # source_URL is the URL where the data can be downloaded from
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config # return an instance of DataIngestionConfig class with the data ingestion configuration

    
        
        

In [15]:
import os
import urllib.request as request
from src.data_science_project_1 import logger
import zipfile

In [16]:
### Component - Data Ingestion Configuration

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config  # config is an instance of DataIngestionConfig class that contains the data ingestion configuration

    def download_file(self):
        # Logic to download data from self.config.source_URL and save it to self.config.local_data_file
        if not os.path.exists(self.config.local_data_file):
            filename, headers =  request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"Downloaded file {filename} with headers {headers}")
        else:
            logger.info(f"File {self.config.local_data_file} already exists. Skipping download.")

    def extract_zip_file(self):
        # Logic to extract the zip file from self.config.local_data_file to self.config.unzip_dir
        
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)  # Ensure the unzip directory exists

        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
            logger.info(f"Extracted files to {unzip_path}")


In [17]:
## FLOW 

try:
    config = ConfigurationManager() # Create an instance of ConfigurationManager to read the configuration files
    data_ingestion_config = config.get_data_ingestion_config() # Get the data ingestion configuration
    data_ingestion = DataIngestion(config=data_ingestion_config) # Create an instance of DataIngestion with the data ingestion configuration
    data_ingestion.download_file() # Download the file from the source URL
    data_ingestion.extract_zip_file() # Extract the downloaded zip file to the specified directory 
except Exception as e:
    logger.exception(f"An error occurred during data ingestion: {e}")
    
    #raise e # If any exception occurs, raise it to be handled by the calling code

[2025-06-20 10:45:17,073: INFO: common: YAML file loaded successfully: config\config.yaml loaded successfully.]
[2025-06-20 10:45:17,076: INFO: common: YAML file loaded successfully: params.yaml loaded successfully.]
[2025-06-20 10:45:17,081: INFO: common: YAML file loaded successfully: schema.yaml loaded successfully.]
[2025-06-20 10:45:17,082: INFO: common: Directory created at: artifacts]
[2025-06-20 10:45:17,083: INFO: common: Directory created at: artifacts/data_ingestion]
[2025-06-20 10:45:17,681: INFO: 641904522: Downloaded file artifacts/data_ingestion/winequality_data.zip with headers Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id